# Zero-Cost LLM & Agent Environment on Kaggle GPU (Ollama + 16K Context)

Use Kaggle's free **30 hours/week GPU quota** (`GPU T4 x2` = 30 GB VRAM) in **either** of two modes:
- **Mode A (100% Inside Kaggle — Recommended if you don't have a strong PC):** Run Ollama, Arize Phoenix (without Docker), and your entire PR Crew code directly inside this notebook using `http://localhost:11434/v1`.
- **Mode B (Remote GPU Server for your Local PC):** Run Cells 1–3 and paste the Cloudflare Quick Tunnel URL into your local `.env`.

### Kaggle Session Settings
1. Set **Accelerator = GPU T4 x2** (or `GPU P100`).
2. Turn **Internet = ON**.

In [ ]:
# Cell 1: Install required system tools (zstd/pciutils needed on Kaggle!) + Ollama + cloudflared
!apt-get update -qq && apt-get install -y -qq zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# Cell 2: Start Ollama with 16K context + 8-bit KV cache (prevents silent 4096-token truncation!)
import os, subprocess, time

MODEL = "qwen3:8b"  # Alternatives: "qwen3.5:4b", "qwen2.5-coder:7b", "qwen3:14b"

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_CONTEXT_LENGTH"] = "16384"
env["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)

!ollama pull {MODEL}
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# Cell 3A (Optional — Mode A: Run 100% Inside Kaggle with In-Process Phoenix, NO Docker needed!)
# Uncomment and run this cell if you want to run your entire agent crew inside Kaggle:
# !git clone https://github.com/PooryaSN2000/mas-hw3-pr-crew.git
# !pip install -q -r mas-hw3-pr-crew/requirements.txt arize-phoenix
# !python mas-hw3-pr-crew/scripts/init_repo.py
# import subprocess
# phoenix_proc = subprocess.Popen(["phoenix", "serve"])
# print("Phoenix started on http://127.0.0.1:6006 | Ollama ready on http://127.0.0.1:11434/v1")

In [ ]:
# Cell 3B (Optional — Mode B: Expose Ollama to your local PC via Cloudflare Quick Tunnel)
import re, subprocess

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        base_url = match.group(0) + "/v1"
        print("=" * 64)
        print("Paste these lines into your local .env file:")
        print(f"LLM_BASE_URL={base_url}")
        print("LLM_API_KEY=ollama")
        print(f"LLM_MODEL={MODEL}")
        print("=" * 64)
        break

tunnel.wait()